# Stage 2 — Baseline models (mathematics)
M1: van der Linden (2007) hierarchical model (2PL + lognormal RT, bivariate person ability/speed).
M2: effort-moderated variant (Wise & DeMars, 2006): NT10 rapid responses treated as missing.
Runtime → Change runtime type → T4 GPU, then Run all. Resumable: finished models are skipped. Output: Drive/PISA2025_stage2.

In [ ]:
!pip -q install numpyro multipledispatch --no-deps
import os, time, json, numpy as np, pandas as pd, jax, jax.numpy as jnp, numpyro, numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from numpyro.diagnostics import summary
import statsmodels.api as sm
from google.colab import drive
drive.mount('/content/drive'); print(jax.devices()); assert jax.devices()[0].platform=='gpu', 'GPU not active: Runtime > Change runtime type > T4 GPU'
S1='/content/drive/MyDrive/PISA2025_stage1'; OUT='/content/drive/MyDrive/PISA2025_stage2'; os.makedirs(OUT,exist_ok=True)
CNTS=['ARM','KAZ','MDA','LVA','LTU','EST','JPN','ARE','SGP']; NOTH=1000; SEED=20251025; STEPS=15000

# ---------- analysis sample (fixed, saved once) ----------
if not os.path.exists(f'{OUT}/sample_math.parquet'):
    rng=np.random.default_rng(SEED); L=[]
    for c in CNTS:
        d=pd.read_parquet(f'{S1}/long_math_{c}.parquet')
        d=d[d.dich & d.valid_t & ~d.trim & ~d.not_reached & d.x.notna()]
        ids=d.CNTSTUID.unique()
        if c!='ARM': ids=rng.choice(ids,size=min(NOTH,len(ids)),replace=False)
        L.append(d[d.CNTSTUID.isin(ids)])
    D=pd.concat(L,ignore_index=True); D.to_parquet(f'{OUT}/sample_math.parquet',index=False)
D=pd.read_parquet(f'{OUT}/sample_math.parquet')
print(D.groupby('CNT').agg(students=('CNTSTUID','nunique'),resp=('x','size'),rapid=('rapid','mean')).round(3))

def model(ii,pp,cp,x,lt,nI,nP,nC):
    a=numpyro.sample('a',dist.LogNormal(0.,.5).expand([nI]))
    b=numpyro.sample('b',dist.Normal(0.,2.).expand([nI]))
    beta=numpyro.sample('beta',dist.Normal(4.,1.).expand([nI]))
    alpha=numpyro.sample('alpha',dist.LogNormal(0.,.5).expand([nI]))
    mu_free=numpyro.sample('mu_free',dist.Normal(0.,1.).expand([nC-1,2]))
    mu=jnp.concatenate([jnp.zeros((1,2)),mu_free],0)                  # reference country (ARM) fixed at 0
    Lc=numpyro.sample('Lcorr',dist.LKJCholesky(2,2.))
    sd_tau=numpyro.sample('sd_tau',dist.HalfNormal(1.))
    sd=jnp.stack([1.,sd_tau])                                           # sd(theta)=1 for identification
    z=numpyro.sample('z',dist.Normal(0.,1.).expand([nP,2]))
    eta=mu[cp]+z@(sd[:,None]*Lc).T
    theta=numpyro.deterministic('theta',eta[:,0]); tau=numpyro.deterministic('tau',eta[:,1])
    numpyro.sample('x',dist.Bernoulli(logits=a[ii]*(theta[pp]-b[ii])),obs=x)
    numpyro.sample('lt',dist.Normal(beta[ii]-tau[pp],1./alpha[ii]),obs=lt)

def fit(tag,d):
    if os.path.exists(f'{OUT}/{tag}_persons.csv'): print(tag,'already done'); return
    ii,items=pd.factorize(d.item); pp,pers=pd.factorize(d.CNTSTUID)
    pc=d.drop_duplicates('CNTSTUID').set_index('CNTSTUID').loc[pers]
    cp=pd.Categorical(pc.CNT,categories=CNTS).codes
    args=dict(ii=jnp.array(ii),pp=jnp.array(pp),cp=jnp.array(cp),x=jnp.array(d.x.values,dtype=jnp.float32),
              lt=jnp.array(np.log(d.t.values),dtype=jnp.float32),nI=len(items),nP=len(pers),nC=len(CNTS))
    from numpyro.infer import SVI, Trace_ELBO, Predictive
    from numpyro.infer.autoguide import AutoNormal
    import optax
    t0=time.time(); guide=AutoNormal(model,init_scale=0.05)
    opt=numpyro.optim.optax_to_numpyro(optax.chain(optax.clip_by_global_norm(10.),optax.adam(optax.exponential_decay(0.02,4000,0.5,end_value=0.001))))
    res=SVI(model,guide,opt,Trace_ELBO()).run(jax.random.PRNGKey(SEED),STEPS,progress_bar=True,**args)
    rt=time.time()-t0; Ls=np.asarray(res.losses); np.save(f'{OUT}/{tag}_elbo.npy',Ls)
    s=dict(Predictive(guide,params=res.params,num_samples=1000)(jax.random.PRNGKey(1),**args))
    s.update(Predictive(model,posterior_samples=s,return_sites=['theta','tau'])(jax.random.PRNGKey(2),**args))
    rmax=float(Ls[-500:].mean()); essmin=float((Ls[-1500:-1000].mean()-Ls[-500:].mean())/abs(Ls[-500:].mean()))
    f=lambda k: np.asarray(s[k])
    pd.DataFrame({'item':items,'a':f('a').mean(0),'b':f('b').mean(0),'beta':f('beta').mean(0),'alpha':f('alpha').mean(0)}).to_csv(f'{OUT}/{tag}_items.csv',index=False)
    P=pc[['CNT','AIUSESCH','ESCS','ST004D01T','EFFORT1','W_FSTUWT','ST438Q01DA','ST438Q02DA','ST438Q03DA','ST438Q04DA']].copy()
    P['theta']=f('theta').mean(0); P['theta_sd']=f('theta').std(0); P['tau']=f('tau').mean(0); P['tau_sd']=f('tau').std(0)
    P.reset_index().to_csv(f'{OUT}/{tag}_persons.csv',index=False)
    mu=np.vstack([np.zeros((1,2)),f('mu_free').mean(0)]); r=float(f('Lcorr')[:,1,0].mean())
    json.dump({'runtime_min':round(rt/60,1),'method':'SVI-AutoNormal','neg_elbo_final':round(rmax,1),'elbo_rel_change_last1000':round(essmin,6),'n_resp':len(d),'n_persons':len(pers),'n_items':len(items),
               'corr_theta_tau':round(r,3),'sd_tau':round(float(f('sd_tau').mean()),3),
               'mu_theta':dict(zip(CNTS,mu[:,0].round(3))),'mu_tau':dict(zip(CNTS,mu[:,1].round(3)))},open(f'{OUT}/{tag}_diag.json','w'),indent=1)
    print(tag,open(f'{OUT}/{tag}_diag.json').read())

fit('M1_vdl',D)
fit('M2_effort',D[~D.rapid])

# ---------- preview: weighted latent-score regression on AI use (country fixed effects) ----------
rows=[]
for tag in ['M1_vdl','M2_effort']:
    P=pd.read_csv(f'{OUT}/{tag}_persons.csv').dropna(subset=['AIUSESCH','ESCS','ST004D01T'])
    P['w']=P.W_FSTUWT/P.groupby('CNT').W_FSTUWT.transform('mean')
    X=pd.get_dummies(P[['AIUSESCH','ESCS','ST004D01T','CNT']],columns=['CNT','ST004D01T'],drop_first=True,dtype=float); X=sm.add_constant(X)
    for y in ['theta','tau']:
        r=sm.WLS(P[y],X,weights=P.w).fit(cov_type='cluster',cov_kwds={'groups':pd.factorize(P.CNT)[0]})
        rows.append({'model':tag,'y':y,'b_AI':r.params['AIUSESCH'],'se':r.bse['AIUSESCH'],'p':r.pvalues['AIUSESCH'],'n':int(r.nobs)})
R=pd.DataFrame(rows).round(4); R.to_csv(f'{OUT}/stage2_ai_preview.csv',index=False); print(R); print('STAGE2 DONE')
